In [1]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 00_environment.ipynb
# Purpose: Configure and validate the execution environment.
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ============================================================
# Project Paths
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
SPLITS_DIR = DATA_DIR / "splits"

MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
UTILS_DIR = PROJECT_ROOT / "utils"
DOCS_DIR = PROJECT_ROOT / "docs"

print(f"Project Root : {PROJECT_ROOT}")

Project Root : /content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM


In [3]:
# ============================================================
# Verify Project Structure
# ============================================================

directories = {
    "Project": PROJECT_ROOT,
    "Raw Data": RAW_DATA_DIR,
    "Processed Data": PROCESSED_DATA_DIR,
    "Splits": SPLITS_DIR,
    "Models": MODELS_DIR,
    "Outputs": OUTPUTS_DIR,
    "Utils": UTILS_DIR,
    "Docs": DOCS_DIR,
}

for name, path in directories.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"{name:<20} {status}")

Project              OK
Raw Data             OK
Processed Data       OK
Splits               OK
Models               OK
Outputs              OK
Utils                OK
Docs                 OK


In [4]:
# ============================================================
# Verify Dataset Files
# ============================================================

quotes_file = RAW_DATA_DIR / "quotes.csv"
labels_file = RAW_DATA_DIR / "gold_standard.csv"

assert quotes_file.exists(), "quotes.csv not found."
assert labels_file.exists(), "gold_standard.csv not found."

print("quotes.csv         ✓")
print("gold_standard.csv  ✓")

quotes.csv         ✓
gold_standard.csv  ✓


In [5]:
# ============================================================
# GPU Information
# ============================================================

import subprocess

try:
    gpu_info = subprocess.check_output(["nvidia-smi"], text=True)
    print(gpu_info)
except Exception:
    print("No NVIDIA GPU detected.")

Sun Jun 28 10:28:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
# ============================================================
# Verify Installed Libraries
# ============================================================

import torch
import transformers
import peft
import trl
import bitsandbytes as bnb
import accelerate
import datasets

print("=" * 60)
print("Environment Summary")
print("=" * 60)

print(f"PyTorch       : {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version  : {torch.version.cuda}")

print(f"Transformers  : {transformers.__version__}")
print(f"PEFT          : {peft.__version__}")
print(f"TRL           : {trl.__version__}")
print(f"BitsAndBytes  : {bnb.__version__}")
print(f"Accelerate    : {accelerate.__version__}")
print(f"Datasets      : {datasets.__version__}")

ValueError: pyarrow.lib.IpcReadOptions size changed, may indicate binary incompatibility. Expected 112 from C header, got 104 from PyObject

In [ ]:
# ============================================================
# Import Required Libraries
# ============================================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

In [ ]:
# ============================================================
# Configure 4-bit Quantization
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
# ============================================================
# Base Model
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(MODEL_NAME)

Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
# ============================================================
# Load Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

print("Tokenizer loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully.


In [ ]:
# ============================================================
# Load Model
# ============================================================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

print("Model loaded successfully.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.


In [ ]:
# ============================================================
# Test Model Inference
# ============================================================

prompt = "What is psychological safety in software engineering?"

messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    text,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.2,
        do_sample=False,
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In the context of software engineering, "psychological safety" refers to an environment where team members feel safe and comfortable expressing their ideas, concerns, and feedback without fear of negative consequences or judgment from others. This concept is crucial for fostering a collaborative and productive work atmosphere.

Psychological safety can be particularly important in agile development practices because it encourages open communication, reduces anxiety among team members, and promotes innovation. It helps build trust within the team, which is essential for effective collaboration and problem-solving.

Key aspects of psychological safety include:

1. **Open Communication**: Encouraging all team members to share their thoughts and ideas freely.
2. **Empathy and Support**: Showing understanding and support for each other's feelings and perspectives.
3. **Res


In [ ]:
# ============================================================
# Environment Ready
# ============================================================

print("=" * 60)
print("Environment is ready.")
print("The project can proceed to 01_baseline.ipynb")
print("=" * 60)

Environment is ready.
The project can proceed to 01_baseline.ipynb
